In [41]:
import os
import json
import re
import pandas as pd
import torch
import numpy as np

from utils import exact_match_results, relax_match_results


In [42]:
# model_folder = 'Output_gemma4_26b_no_sim'
# model_folder = "Output_llama3.1_8b_no_sim"
model_folder = "Output_mistral-small_24b_no_sim"
# model_folder = "Output_gemma4_31b_no_sim"

# model_sim_folder = 'Output_llama3.1_8b_sim'
# model_sim_folder = "Output_gemma4_31b_sim"
model_sim_folder = "Output_mistral-small_24b_sim"
# model_sim_folder = "Output_gemma4_26b_sim"

if os.path.exists(f"{model_sim_folder}") == False:
    os.makedirs(f"{model_sim_folder}")

if os.path.exists(f"{model_sim_folder}") == False:
    os.makedirs(f"{model_sim_folder}")

In [43]:
# Checking if there's any null output:
for sample in os.listdir(model_folder):
    with open(f"{model_folder}/{sample}", "r") as f:
        data = json.load(f)
        if not data:
            print(f"Sample {sample} has null output.")


Sample sample_30_20210610044412_Isra has null output.


In [44]:
preds_dict_od_all = []
preds_dict_all_temp = []
for sample in os.listdir(model_folder):
    # if sample in ['sample_68', 'sample_635', 'sample_71', 'sample_324','sample_2748']:
        print(f"evaluating {sample}:")
        with open(f'label/{sample}.json', 'r') as f:
            label_dict = json.load(f)

        #Read output
        with open(f"{model_folder}/{sample}", 'r') as f:
            preds_dict = json.load(f)
        # print(preds_dict)
        #Read original text
        with open(f'notes/{sample}.txt', 'r') as file:
            text = file.read()
        preds_dict_all_temp.append(preds_dict)
        preds_dict_processed = []
        preds_dict_od = []
        for ent in preds_dict:
            for phrase, label in ent.items():
        
                matches = list(re.finditer(re.escape(phrase), text,re.IGNORECASE))
                if matches:
                    for match in matches:
                        # print(f"Found '{match.group()}' at [{match.start()}, {match.end()}]")
                        preds_dict_processed.append({'entity':match.group(), 'label': label, 'start': match.start(), 'end': match.end()})
                else:

                    preds_dict_od.append({'entity':phrase, 'label': 'OD', 'start': 0, 'end': 0})
                    preds_dict_od_all.append({'sample':sample,'entity':phrase, 'label': 'OD', 'start': 0, 'end': 0})
        # with open(f"{model_attribution_folder}/{sample}", 'w') as f:
        #     json.dump(preds_dict_od, f, indent=2)

len(preds_dict_od_all)

evaluating sample_134_20210112211711:
evaluating sample_25_20210610044410_Isra:
evaluating sample_38_20210610044413_Hayden:
evaluating sample_323:
evaluating sample_169_20210112211715:
evaluating sample_1246:
evaluating sample_46_20210610044418_Hayden:
evaluating sample_2789:
evaluating sample_116_20210112211709:
evaluating sample_1262:
evaluating sample_2764:
evaluating sample_76_20210112211704:
evaluating sample_37_20210621151709_Hayden:
evaluating sample_38_20210604143816_Isra:
evaluating sample_2757:
evaluating sample_80_20210604143825_Booma:
evaluating sample_367:
evaluating sample_63_20210112211702:
evaluating sample_324:
evaluating sample_104_20210112211707:
evaluating sample_225:
evaluating sample_160_20210112211715:
evaluating sample_364:
evaluating sample_30_20210610044412_Isra:
evaluating sample_80_20210112211705:
evaluating sample_88_20210604143828_Booma:
evaluating sample_2_20210604143805_Hayden:
evaluating sample_23_20210610044409_Isra:
evaluating sample_380:
evaluating s

34

In [51]:
dedup = []
for item in preds_dict_od_all:
    if item not in dedup:
        dedup.append(item)

print(len(dedup))
dedup

33


[{'sample': 'sample_134_20210112211711',
  'entity': 'hearing problems',
  'label': 'OD',
  'start': 0,
  'end': 0},
 {'sample': 'sample_134_20210112211711',
  'entity': 'skin changes',
  'label': 'OD',
  'start': 0,
  'end': 0},
 {'sample': 'sample_134_20210112211711',
  'entity': 'problems with hormones',
  'label': 'OD',
  'start': 0,
  'end': 0},
 {'sample': 'sample_25_20210610044410_Isra',
  'entity': 'cranial nervous systems',
  'label': 'OD',
  'start': 0,
  'end': 0},
 {'sample': 'sample_98_20210112211707',
  'entity': 'throat pain',
  'label': 'OD',
  'start': 0,
  'end': 0},
 {'sample': 'sample_98_20210112211707',
  'entity': 'muscle tenderness',
  'label': 'OD',
  'start': 0,
  'end': 0},
 {'sample': 'sample_98_20210112211707',
  'entity': 'loss of sensation',
  'label': 'OD',
  'start': 0,
  'end': 0},
 {'sample': 'sample_98_20210112211707',
  'entity': 'excessive appetite',
  'label': 'OD',
  'start': 0,
  'end': 0},
 {'sample': 'sample_157_20210112211713',
  'entity': 'po

# Similarity algorithm

In [46]:
# from sentence_transformers import SentenceTransformer, util
# model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# for sample in os.listdir(model_folder):
#     # if sample == 'sample_58_20210604143820_Isra':
#     #Read original text
#     #Read label dict
#         print(f"processing: {sample}")

#         with open(f'label/{sample}.json', 'r') as f:
#             label_dict = json.load(f)
            
#         #Read output
#         with open(f"{model_folder}/{sample}", 'r') as f:
#             org_preds_json = json.load(f)

#         #Read original text
#         with open(f'notes/{sample}.txt', 'r') as file:
#             text = file.read()

#         preds_dict = []
#         messup_pred_tokens = []
#         for ent in org_preds_json:
#             for phrase, label in ent.items():
#                 matches = list(re.finditer(re.escape(phrase), text, re.IGNORECASE))
#                 if matches:
#                     for match in matches:
#                         # print(f"Found '{match.group()}' at [{match.start()}, {match.end()}]")
#                         preds_dict.append({'entity':match.group(), 'label': label, 'start': match.start(), 'end': match.end()})
#                 else:
#                     messup_pred_tokens.append(phrase)
        
        
#         clean_text = re.sub(r'[.,:&]', '', text)
#         clean_original_tokens = clean_text.split()

#         # Encode all candidate spans
#         n = len(clean_original_tokens)
#         print(f"length of the text: {n}")


#         spans_dup = []
#         for i in range(0, n):
#             for w in range(1, n+1):
#                 spans_dup.append(' '.join(clean_original_tokens[i:i+w]))
#         spans = list(set(spans_dup)) #Deduplication

#         #Embedding
#         spans_emb = model.encode(spans)

#         alg_pred_tokens = []
#         for token in messup_pred_tokens:    
#             # Encode target
#             target_emb = model.encode(token)
#             sim = util.cos_sim(target_emb, spans_emb) 

#             #Mapping back all values = highest value
#             top_idx = sim.argmax()

#             #Map back embedded vector to tokens
#             best_match = spans[top_idx]
#             alg_pred_tokens.append(best_match)

#         #Write back the output
#         mapping = dict(zip(messup_pred_tokens, alg_pred_tokens))

#         original = []
#         updated = []

#         for record in org_preds_json:
#             original_record = {}
#             updated_record = {}

#             for key, value in record.items():

#                 if key in mapping:
#                     original_record[key] = "false positive"
#                     updated_record[mapping[key]] = value
                    
#                 else:
#                     original_record[key] = value
#                     updated_record[key] = value

#             original.append(original_record)
#             updated.append(updated_record)
        
#         # # with open(f"{model_no_sim_folder}/{sample}", 'w') as f:
#         # #     json.dump(original, f, indent=2)
#         # with open(f"{model_sim_folder}/{sample}", 'w') as f:
#         #     json.dump(updated, f, indent=2)


In [47]:
from sentence_transformers import SentenceTransformer, util
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)
print(f"Using device: {device}")

for sample in os.listdir(model_folder):
    # if sample == 'sample_134_20210112211711':
    #Read original text
    #Read label dict
        print(f"processing: {sample}")

        with open(f'label/{sample}.json', 'r') as f:
            label_dict = json.load(f)
            
        #Read output
        with open(f"{model_folder}/{sample}", 'r') as f:
            org_preds_json = json.load(f)

        #Read original text
        with open(f'notes/{sample}.txt', 'r') as file:
            text = file.read()

        preds_dict = []
        messup_pred_tokens = []
        for ent in org_preds_json:
            for phrase, label in ent.items():
                matches = list(re.finditer(re.escape(phrase), text, re.IGNORECASE))
                if matches:
                    for match in matches:
                        # print(f"Found '{match.group()}' at [{match.start()}, {match.end()}]")
                        preds_dict.append({'entity':match.group(), 'label': label, 'start': match.start(), 'end': match.end()})
                else:
                    messup_pred_tokens.append(phrase)
        
        # print(messup_pred_tokens)
        
        clean_text = re.sub(r'[.,:&]', '', text)
        clean_original_tokens = clean_text.split()

        # Encode all candidate spans
        n = len(clean_original_tokens)
        print(f"length of the text: {n}")


        unique_messup_pred_tokens = list(dict.fromkeys(messup_pred_tokens))
        max_target_length = max((len(token.split()) for token in unique_messup_pred_tokens if token.strip()), default=0)
        target_lengths = range(1, max_target_length*5)

        spans = []
        seen_spans = set()
        for i in range(n):
            remaining = n - i
            for width in target_lengths:
                if width > remaining:
                    break
                span = ' '.join(clean_original_tokens[i:i + width])
                if span not in seen_spans:
                    seen_spans.add(span)
                    spans.append(span)

        if spans and unique_messup_pred_tokens:
            spans_emb = model.encode(
                spans,
                batch_size=128,
                convert_to_tensor=True,
                normalize_embeddings=True,
                show_progress_bar=False,
            )

            target_emb = model.encode(
                unique_messup_pred_tokens,
                batch_size=128,
                convert_to_tensor=True,
                normalize_embeddings=True,
                show_progress_bar=False,
            )

            sim = util.cos_sim(target_emb, spans_emb)
            top_indices = sim.argmax(dim=1).tolist()
            resolved_tokens = [spans[idx] for idx in top_indices]
            resolved_mapping = dict(zip(unique_messup_pred_tokens, resolved_tokens))
            alg_pred_tokens = [resolved_mapping[token] for token in messup_pred_tokens]
        else:
            alg_pred_tokens = []

        #Write back the output
        mapping = dict(zip(messup_pred_tokens, alg_pred_tokens))

        original = []
        updated = []

        for record in org_preds_json:
            original_record = {}
            updated_record = {}

            for key, value in record.items():

                if key in mapping:
                    original_record[key] = "false positive"
                    updated_record[mapping[key]] = value
                    
                else:
                    original_record[key] = value
                    updated_record[key] = value

            original.append(original_record)
            updated.append(updated_record)
        
        # # with open(f"{model_no_sim_folder}/{sample}", 'w') as f:
        # #     json.dump(original, f, indent=2)
        with open(f"{model_sim_folder}/{sample}", 'w') as f:
            json.dump(updated, f, indent=2)


Using device: cuda
processing: sample_134_20210112211711
length of the text: 305
processing: sample_25_20210610044410_Isra
length of the text: 372
processing: sample_38_20210610044413_Hayden
length of the text: 540
processing: sample_323
length of the text: 279
processing: sample_169_20210112211715
length of the text: 801
processing: sample_1246
length of the text: 336
processing: sample_46_20210610044418_Hayden
length of the text: 228
processing: sample_2789
length of the text: 767
processing: sample_116_20210112211709
length of the text: 210
processing: sample_1262
length of the text: 816
processing: sample_2764
length of the text: 415
processing: sample_76_20210112211704
length of the text: 241
processing: sample_37_20210621151709_Hayden
length of the text: 285
processing: sample_38_20210604143816_Isra
length of the text: 511
processing: sample_2757
length of the text: 394
processing: sample_80_20210604143825_Booma
length of the text: 233
processing: sample_367
length of the text: 5

In [48]:
target_lengths

range(1, 0)

In [49]:
tp_rm, fp_rm, fn_rm, od_rm = 0, 0, 0, 0
tp_em, fp_em, fn_em, od_em = 0, 0, 0, 0
for sample in os.listdir(model_sim_folder):
    # if sample not in ['sample_140_20210112211712']:
        print(f"evaluating {sample}:")
        with open(f'label/{sample}.json', 'r') as f:
            label_dict = json.load(f)

        #Read output
        with open(f"{model_sim_folder}/{sample}", 'r') as f:
            preds_dict = json.load(f)
        # print(preds_dict)
        #Read original text
        with open(f'notes/{sample}.txt', 'r') as file:
            text = file.read()

        preds_dict_processed = []
        for ent in preds_dict:
            for phrase, label in ent.items():
        
                matches = list(re.finditer(re.escape(phrase), text,re.IGNORECASE))
                if matches:
                    for match in matches:
                        # print(f"Found '{match.group()}' at [{match.start()}, {match.end()}]")
                        preds_dict_processed.append({'entity':match.group(), 'label': label, 'start': match.start(), 'end': match.end()})
                else:
                    preds_dict_processed.append({'entity':phrase, 'label': 'OD', 'start': 0, 'end': 0})
                
        #Deduplication
        preds_dict_final = []
        seen = set()
        for d in preds_dict_processed:
            t = tuple(sorted(d.items()))
            if t not in seen:
                seen.add(t)
                preds_dict_final.append(d)
                
        ## Evaluation
        em, em_mm, em_ud, em_od = exact_match_results(label_dict, preds_dict_final, 'Output_verification', sample)
        rm, rm_mm, rm_ud, rm_od = relax_match_results(label_dict, preds_dict_final, 'Output_verification', sample)
    
        tp_rm+= rm
        fp_rm+= rm_mm
        fn_rm+= rm_ud
        od_rm+= rm_od

            
        tp_em+= em
        fp_em+= em_mm
        fn_em+= em_ud
        od_em+= em_od

rm_precision = tp_rm/(tp_rm + rm_mm + od_rm)
rm_recall = tp_rm/(tp_rm + fn_rm)

em_precision = tp_em/(tp_em + em_mm + od_em)
em_recall = tp_em/(tp_em + fn_em)

print(f"SUMMARY EXACT MATCH: TP {tp_em}, FP {fp_em}, UD {fn_em}, OD {od_em}")
print(f"MICRO AVERAGE SCORE EXACT MATCH: Precision: {em_precision}, Recall: {em_recall}, F1 score: {2*em_precision*em_recall/(em_precision + em_recall)}")

print(f"SUMMARY RELAX MATCH: TP {tp_rm}, FP {fp_rm}, UD {fn_rm}, OD {od_rm}")
print(f"MICRO AVERAGE SCORE RELAX MATCH: Precision: {rm_precision}, Recall: {rm_recall}, F1 score: {2*rm_precision*rm_recall/(rm_precision + rm_recall)}")

evaluating sample_134_20210112211711:
[{'entity': 'balance problems', 'label': 'PROBLEM', 'start': 467, 'end': 483}, {'entity': 'pain', 'label': 'PROBLEM', 'start': 1190, 'end': 1194}, {'entity': 'pain', 'label': 'PROBLEM', 'start': 1633, 'end': 1637}, {'entity': 'pain', 'label': 'PROBLEM', 'start': 1645, 'end': 1649}, {'entity': 'bleeding', 'label': 'PROBLEM', 'start': 611, 'end': 619}, {'entity': 'bleeding', 'label': 'PROBLEM', 'start': 1380, 'end': 1388}, {'entity': 'UTI', 'label': 'PROBLEM', 'start': 245, 'end': 248}, {'entity': 'nipple changes', 'label': 'PROBLEM', 'start': 1430, 'end': 1444}, {'entity': 'problems with cholesterol', 'label': 'PROBLEM', 'start': 1967, 'end': 1992}, {'entity': 'hormones', 'label': 'PROBLEM', 'start': 1996, 'end': 2004}, {'entity': 'cancer', 'label': 'PROBLEM', 'start': 1572, 'end': 1578}]
[{'entity': 'glasses', 'label': 'PROBLEM', 'start': 350, 'end': 357}, {'entity': 'dentures', 'label': 'PROBLEM', 'start': 598, 'end': 606}]
EM sample_134_202101122

In [50]:
tp_rm, fp_rm, fn_rm, od_rm = 0, 0, 0, 0
tp_em, fp_em, fn_em, od_em = 0, 0, 0, 0
for sample in os.listdir(model_folder):
    # if sample not in ['sample_140_20210112211712']:
        print(f"evaluating {sample}:")
        with open(f'label/{sample}.json', 'r') as f:
            label_dict = json.load(f)

        #Read output
        with open(f"{model_folder}/{sample}", 'r') as f:
            preds_dict = json.load(f)
        # print(preds_dict)
        #Read original text
        with open(f'notes/{sample}.txt', 'r') as file:
            text = file.read()

        preds_dict_processed = []
        for ent in preds_dict:
            for phrase, label in ent.items():
        
                matches = list(re.finditer(re.escape(phrase), text,re.IGNORECASE))
                if matches:
                    for match in matches:
                        # print(f"Found '{match.group()}' at [{match.start()}, {match.end()}]")
                        preds_dict_processed.append({'entity':match.group(), 'label': label, 'start': match.start(), 'end': match.end()})
                else:
                    preds_dict_processed.append({'entity':phrase, 'label': 'OD', 'start': 0, 'end': 0})
                
        #Deduplication
        preds_dict_final = []
        seen = set()
        for d in preds_dict_processed:
            t = tuple(sorted(d.items()))
            if t not in seen:
                seen.add(t)
                preds_dict_final.append(d)
                
        ## Evaluation
        em, em_mm, em_ud, em_od = exact_match_results(label_dict, preds_dict_final, 'Output_verification', sample)
        rm, rm_mm, rm_ud, rm_od = relax_match_results(label_dict, preds_dict_final, 'Output_verification', sample)
    
        tp_rm+= rm
        fp_rm+= rm_mm
        fn_rm+= rm_ud
        od_rm+= rm_od

            
        tp_em+= em
        fp_em+= em_mm
        fn_em+= em_ud
        od_em+= em_od

rm_precision = tp_rm/(tp_rm + rm_mm + od_rm)
rm_recall = tp_rm/(tp_rm + fn_rm)

em_precision = tp_em/(tp_em + em_mm + od_em)
em_recall = tp_em/(tp_em + fn_em)

print(f"SUMMARY EXACT MATCH: TP {tp_em}, FP {fp_em}, UD {fn_em}, OD {od_em}")
print(f"MICRO AVERAGE SCORE EXACT MATCH: Precision: {em_precision}, Recall: {em_recall}, F1 score: {2*em_precision*em_recall/(em_precision + em_recall)}")

print(f"SUMMARY RELAX MATCH: TP {tp_rm}, FP {fp_rm}, UD {fn_rm}, OD {od_rm}")
print(f"MICRO AVERAGE SCORE RELAX MATCH: Precision: {rm_precision}, Recall: {rm_recall}, F1 score: {2*rm_precision*rm_recall/(rm_precision + rm_recall)}")

evaluating sample_134_20210112211711:
[{'entity': 'hearing problems', 'label': 'OD', 'start': 0, 'end': 0}, {'entity': 'balance problems', 'label': 'PROBLEM', 'start': 467, 'end': 483}, {'entity': 'pain', 'label': 'PROBLEM', 'start': 1190, 'end': 1194}, {'entity': 'pain', 'label': 'PROBLEM', 'start': 1633, 'end': 1637}, {'entity': 'pain', 'label': 'PROBLEM', 'start': 1645, 'end': 1649}, {'entity': 'bleeding', 'label': 'PROBLEM', 'start': 611, 'end': 619}, {'entity': 'bleeding', 'label': 'PROBLEM', 'start': 1380, 'end': 1388}, {'entity': 'UTI', 'label': 'PROBLEM', 'start': 245, 'end': 248}, {'entity': 'skin changes', 'label': 'OD', 'start': 0, 'end': 0}, {'entity': 'nipple changes', 'label': 'PROBLEM', 'start': 1430, 'end': 1444}, {'entity': 'problems with cholesterol', 'label': 'PROBLEM', 'start': 1967, 'end': 1992}, {'entity': 'problems with hormones', 'label': 'OD', 'start': 0, 'end': 0}, {'entity': 'cancer', 'label': 'PROBLEM', 'start': 1572, 'end': 1578}]
[{'entity': 'glasses', 'la